# Optuna — Hyperparameter sweep with budget discipline

## Project goal

Tune a LightGBM vol forecaster with TPE + MedianPruner inside `TimeSeriesSplit` CV. Persist the study to SQLite (so it's resumable). Visualise the search. Add a multi-objective version that trades off MAE against fit time.


## Why this exercises the cheatsheet

Optuna is most often used badly: 100 trials on the same data twice, no pruning, no persistence, results never inspected. This project does it right — every feature gets exercised in a way that mirrors a real production tuning run.


## Sub-task 1: Define the objective

Write `objective(trial)` that fits a LightGBM regressor through `TimeSeriesSplit(5).split(X_train)`, computes mean fold MAE, and returns it. Search space: `n_estimators ∈ [100, 400]`, `learning_rate ∈ [0.01, 0.2]` (log scale), `num_leaves ∈ [15, 127]`, `min_child_samples ∈ [5, 50]`.

**Patterns this forces:**

- `trial.suggest_int / suggest_float / suggest_loguniform`
- `TimeSeriesSplit(5).split(X_train)` inside the objective
- `np.mean(fold_maes)` as the return


In [ ]:
# Your answer here

## Sub-task 2: Run with TPE + MedianPruner

Create a study with `sampler=TPESampler(seed=0)` and `pruner=MedianPruner(n_startup_trials=5)`. Inside the objective, call `trial.report(mean_so_far, step=fold_k)` and `if trial.should_prune(): raise TrialPruned()`. Run 30 trials. Print best params and pruned-trial count.

**Patterns this forces:**

- `optuna.create_study(sampler=..., pruner=..., direction='minimize')`
- `trial.report(...)` after each fold
- `if trial.should_prune(): raise optuna.TrialPruned()`


In [ ]:
# Your answer here

## Sub-task 3: Persist to SQLite

Re-run the study with `storage='sqlite:///<path>.db', study_name='vol_forecaster_v1'`. Stop after 5 trials. Reload via `optuna.load_study(...)` and continue for another 5 trials. Confirm the trials accumulated.

**Patterns this forces:**

- `storage='sqlite:///path.db'`
- `optuna.load_study(study_name=..., storage=...)`
- `load_if_exists=True` for resume-or-create


In [ ]:
# Your answer here

## Sub-task 4: Visualise the search

Use `optuna.visualization`: `plot_optimization_history`, `plot_param_importances`, `plot_parallel_coordinate`. (Requires `pip install plotly`.) Identify the most-important parameter.

**Patterns this forces:**

- `optuna.visualization.plot_optimization_history(study)`
- `plot_param_importances(study)`
- interpretation: which knob mattered most


In [ ]:
# Your answer here

## Sub-task 5: Inspect via trials_dataframe

Extract `study.trials_dataframe()`. Filter to only completed trials. Compute the empirical correlation between each parameter and the objective. Compare to the param-importance plot from sub-task 4.

**Patterns this forces:**

- `study.trials_dataframe()`
- filter by `state == 'COMPLETE'`
- `df.corr()` on `[params_*, value]`


In [ ]:
# Your answer here

## Sub-task 6: Multi-objective: MAE vs fit time

Rewrite the objective to return a tuple `(mae, fit_seconds)`. Create a study with `directions=['minimize', 'minimize']`. Run 20 trials. Inspect `study.best_trials` (the Pareto front) — print the trade-off for the cheapest, fastest, and balanced trials.

**Patterns this forces:**

- objective returning `(mae, fit_time)`
- `directions=['minimize', 'minimize']`
- `study.best_trials` (NOT `best_trial`) — Pareto-optimal set


In [ ]:
# Your answer here

## What success looks like

- A SQLite-backed study of 30+ trials with at least 30% pruned.
- Plotly figures showing the search history and parameter importance.
- A 2-3 row Pareto table from the multi-objective run.
- Best params reproducibly load from disk via `optuna.load_study`.
- One sentence: *'after this search, the model's val MAE improved by X%; learning_rate mattered most.'*
